# Rename Model Values: Flat → Pipe-Delimited

**Prerequisite:** `migrate_schema.ipynb` must be run first (renames the `channel` column to `model`).

This notebook transforms the old flat model name values in the `model` column
to match the pipe-delimited naming convention used in the 202607+ schema:

```
Search CVR Google Brand  →  Search | Conversion | Brand
Social AWE Meta NA       →  Social | Awareness | AWE Social Meta
```

Files whose `model` values already contain `|` are automatically skipped.

Run cells in order.

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

# Add project root so joblib can deserialize custom model classes
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import functions.lr_models  # noqa: F401

OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

print(f"Output dir: {OUTPUT_DIR}")

Output dir: /Users/jeff.parks/Dev/revenue-forecasting/data/output


## Mapping Table

Edit this cell if you need to add or adjust any mappings before applying.

In [2]:
MODEL_NAME_MAP = {
    # Search
    "Search CVR Google Brand":    "Search | Conversion | Brand",
    "Search CVR Google NonBrand": "Search | Conversion | Non Brand",
    "Search CVR Google PMAX":     "Search | Conversion | PMAX",
    "Search CVR Google RSC":      "Search | Conversion | RSC",
    "Search CON Google NonBrand": "Search | Consideration | Non Brand",
    # Social — Awareness
    "Social AWE Meta NA":         "Social | Awareness | AWE Social Meta",
    "Social AWE Pinterest NA":    "Social | Awareness | AWE Social Pinterest",
    "Social AWE TikTok NA":       "Social | Awareness | AWE Social TikTok",
    # Social — Consideration
    "Social CON Meta NA":         "Social | Consideration | CON Social Meta",
    "Social CON Pinterest NA":    "Social | Consideration | CON Social Pinterest",
    # Social — Conversion
    "Social CVR Meta ASC Omni":   "Social | Conversion | CVR ASC Omni",
    "Social CVR Meta ASC Value":  "Social | Conversion | CVR ASC Value",
    "Social CVR Meta ASC Volume": "Social | Conversion | CVR ASC Volume",
}

print(f"{len(MODEL_NAME_MAP)} mappings defined.")
display(
    pd.DataFrame(
        [(k, v) for k, v in MODEL_NAME_MAP.items()],
        columns=["Old name (flat)", "New name (pipe-delimited)"]
    )
)

13 mappings defined.


,Old name (flat),New name (pipe-delimited)
0,Search CVR Google Brand,Search | Conversion | Brand
1,Search CVR Google NonBrand,Search | Conversion | Non Brand
2,Search CVR Google PMAX,Search | Conversion | PMAX
3,Search CVR Google RSC,Search | Conversion | RSC
4,Search CON Google NonBrand,Search | Consideration | Non Brand
5,Social AWE Meta NA,Social | Awareness | AWE Social Meta
6,Social AWE Pinterest NA,Social | Awareness | AWE Social Pinterest
7,Social AWE TikTok NA,Social | Awareness | AWE Social TikTok
8,Social CON Meta NA,Social | Consideration | CON Social Meta
9,Social CON Pinterest NA,Social | Consideration | CON Social Pinterest


## 1. Discovery

Find which joblib files still contain flat model names (no `|` separator).

In [3]:
needs_rename = []
already_pipe = []
unmapped_by_file = {}

for path in sorted(OUTPUT_DIR.glob("models_*.joblib")):
    df = joblib.load(path)
    unique_models = df["model"].unique().tolist()
    flat_values = [v for v in unique_models if "|" not in str(v)]

    if not flat_values:
        already_pipe.append(path)
    else:
        needs_rename.append(path)
        unmapped = [v for v in flat_values if v not in MODEL_NAME_MAP]
        if unmapped:
            unmapped_by_file[path.name] = unmapped

print(f"{'File':<40} {'Status'}")
print("-" * 60)
for p in needs_rename:
    print(f"{p.name:<40} needs rename")
for p in already_pipe:
    print(f"{p.name:<40} already pipe-delimited — skip")

print(f"\n{len(needs_rename)} file(s) to process, {len(already_pipe)} already up to date.")

if unmapped_by_file:
    print("\n⚠ UNMAPPED values found — add them to MODEL_NAME_MAP before applying:")
    for fname, vals in unmapped_by_file.items():
        print(f"  {fname}: {vals}")
else:
    print("\n✓ All flat model names are covered by the mapping table.")

File                                     Status
------------------------------------------------------------
models_202603_3P.joblib                  needs rename
models_202603_reflows.joblib             needs rename
models_202604_1P.joblib                  needs rename
models_202604_2P.joblib                  needs rename
models_202604_3P.joblib                  needs rename
models_202604_BRFS_flat.joblib           needs rename
models_202604_mmm.joblib                 needs rename
models_202605_1P.joblib                  needs rename
models_202605_2P.joblib                  needs rename
models_202605_3P.joblib                  needs rename
models_202606_1P.joblib                  needs rename
models_202606_2P.joblib                  already pipe-delimited — skip
models_202607_1P.joblib                  already pipe-delimited — skip
models_202607_2P.joblib                  already pipe-delimited — skip

11 file(s) to process, 3 already up to date.

✓ All flat model names are covered by

## 2. Preview (dry run)

Show the exact before → after transformation for one file. Nothing is written.

In [4]:
if not needs_rename:
    print("Nothing to preview — all files are already pipe-delimited.")
else:
    sample = joblib.load(needs_rename[0])
    unique_before = sorted(sample["model"].unique())
    preview_rows = [
        {"Before": v, "After": MODEL_NAME_MAP.get(v, f"⚠ UNMAPPED: {v}")}
        for v in unique_before
    ]
    print(f"Preview for: {needs_rename[0].name}")
    display(pd.DataFrame(preview_rows))

Preview for: models_202603_3P.joblib


,Before,After
0,Search CON Google NonBrand,Search | Consideration | Non Brand
1,Search CVR Google Brand,Search | Conversion | Brand
2,Search CVR Google NonBrand,Search | Conversion | Non Brand
3,Search CVR Google PMAX,Search | Conversion | PMAX
4,Search CVR Google RSC,Search | Conversion | RSC
5,Social AWE Meta NA,Social | Awareness | AWE Social Meta
6,Social AWE Pinterest NA,Social | Awareness | AWE Social Pinterest
7,Social AWE TikTok NA,Social | Awareness | AWE Social TikTok
8,Social CON Meta NA,Social | Consideration | CON Social Meta
9,Social CON Pinterest NA,Social | Consideration | CON Social Pinterest


## 3. Apply

Rename model values in each file and overwrite both the `.joblib` and companion `.csv`.

> Originals were backed up by `migrate_schema.ipynb`. If you need a fresh backup first, re-run that notebook's backup cell.

In [5]:
if unmapped_by_file:
    print("⛔ Stopping — unmapped values exist. Add them to MODEL_NAME_MAP and re-run.")
elif not needs_rename:
    print("Nothing to apply — all files already use pipe-delimited names.")
else:
    for joblib_path in needs_rename:
        df = joblib.load(joblib_path)
        df["model"] = df["model"].map(lambda v: MODEL_NAME_MAP.get(v, v))

        joblib.dump(df, joblib_path)

        csv_path = joblib_path.with_suffix(".csv")
        df.drop(columns=["model_obj"], errors="ignore").to_csv(csv_path, index=False)

        print(f"✓  {joblib_path.name}")

    print(f"\nRenamed model values in {len(needs_rename)} file(s).")

✓  models_202603_3P.joblib
✓  models_202603_reflows.joblib
✓  models_202604_1P.joblib
✓  models_202604_2P.joblib
✓  models_202604_3P.joblib
✓  models_202604_BRFS_flat.joblib
✓  models_202604_mmm.joblib
✓  models_202605_1P.joblib
✓  models_202605_2P.joblib
✓  models_202605_3P.joblib
✓  models_202606_1P.joblib

Renamed model values in 11 file(s).


## 4. Apply — Results (CSV only)

Rename `model` values in each `results_*.csv` file. Files that are already
pipe-delimited are skipped automatically.

In [9]:
RESULTS_COL_RENAME = {
    "channel":    "model",
    "model":      "estimator",
    "base_model": "base_estimator",
}

results_to_update = []
results_skip = []

for path in sorted(OUTPUT_DIR.glob("results_*.csv")):
    df = pd.read_csv(path)
    has_old_cols = "channel" in df.columns
    has_model_col = "model" in df.columns
    if has_old_cols:
        results_to_update.append(path)
    elif has_model_col:
        flat = [v for v in df["model"].unique() if "|" not in str(v)]
        if flat:
            results_to_update.append(path)
        else:
            results_skip.append((path, "already pipe-delimited"))
    else:
        results_skip.append((path, "no model/channel column"))

for path in results_to_update:
    df = pd.read_csv(path)

    # Step 1: rename columns if old schema
    if "channel" in df.columns:
        actual = {k: v for k, v in RESULTS_COL_RENAME.items() if k in df.columns}
        df = df.rename(columns=actual)

    # Step 2: rename flat model values
    unmapped = [v for v in df["model"].unique() if "|" not in str(v) and v not in MODEL_NAME_MAP]
    if unmapped:
        print(f"⚠  {path.name} — unmapped model values (skipping): {unmapped}")
        continue

    df["model"] = df["model"].map(lambda v: MODEL_NAME_MAP.get(v, v))
    df.to_csv(path, index=False)
    print(f"✓  {path.name}")

for path, reason in results_skip:
    print(f"   skipped ({reason}): {path.name}")

print(f"\nUpdated {len(results_to_update)} results file(s).")

✓  results_202603_3P.csv
✓  results_202603_reflows.csv
✓  results_202604_1P.csv
✓  results_202604_1P_marchnew.csv
✓  results_202604_1P_marchorig.csv
✓  results_202604_2P.csv
✓  results_202604_3P.csv
✓  results_202604_BRFS_flat.csv
✓  results_202604_mmm.csv
✓  results_202605_1P.csv
✓  results_202605_2P.csv
✓  results_202605_3P.csv
✓  results_202606_1P.csv
   skipped (already pipe-delimited): results_202606_2P.csv
   skipped (already pipe-delimited): results_202607_1P.csv
   skipped (already pipe-delimited): results_202607_2P.csv

Updated 13 results file(s).


## 5. Verify

Reload all files and confirm every `model` value is now pipe-delimited across
both models joblib files and results CSVs.

In [11]:
issues = []

for path in sorted(OUTPUT_DIR.glob("models_*.joblib")):
    df = joblib.load(path)
    flat = [v for v in df["model"].unique() if "|" not in str(v)]
    if flat:
        issues.append((path.name, flat))

for path in sorted(OUTPUT_DIR.glob("results_*.csv")):
    df = pd.read_csv(path)
    if "model" not in df.columns:
        continue
    flat = [v for v in df["model"].unique() if "|" not in str(v)]
    if flat:
        issues.append((path.name, flat))

if issues:
    print("⚠ Flat model names still present:")
    for name, vals in issues:
        print(f"  {name}: {vals}")
else:
    model_files = sorted(OUTPUT_DIR.glob("models_*.joblib"))
    results_files = sorted(OUTPUT_DIR.glob("results_*.csv"))
    print(f"✓ All {len(model_files)} models file(s) and {len(results_files)} results file(s) verified — every model value is pipe-delimited.")
    print()

    all_models = set()
    for path in model_files:
        df = joblib.load(path)
        all_models.update(df["model"].unique())
    for path in results_files:
        df = pd.read_csv(path)
        if "model" in df.columns:
            all_models.update(v for v in df["model"].unique() if "|" in str(v))

    print("All unique pipe-delimited model names:")
    for m in sorted(all_models):
        print(f"  {m}")

✓ All 14 models file(s) and 16 results file(s) verified — every model value is pipe-delimited.

All unique pipe-delimited model names:
  Search | Consideration | Non Brand
  Search | Conversion | Brand
  Search | Conversion | Non Brand
  Search | Conversion | PMAX
  Search | Conversion | RSC
  Social | Awareness | AWE Social META Custom
  Social | Awareness | AWE Social Meta
  Social | Awareness | AWE Social Pinterest
  Social | Awareness | AWE Social TikTok
  Social | Consideration | CON Social META Custom
  Social | Consideration | CON Social Meta
  Social | Consideration | CON Social Pinterest
  Social | Conversion | CVR ASC Omni
  Social | Conversion | CVR ASC Value
  Social | Conversion | CVR ASC Volume
  Social | Conversion | CVR Social META ASC Max volume
  Social | Conversion | CVR Social Meta ASC Max Value
  Social | Conversion | CVR Social Meta ASC Max Volume
  Social | Conversion | CVR Social Meta ASC Omni
